In [ ]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import pandas as pd

file_path = "/content/drive/MyDrive/AI_Email_Deliverability_Intelligence/data/raw/SendGuard900K.csv"

schema_df = pd.read_csv(
    file_path,
    nrows=1000,
    low_memory=False
)

schema_report = pd.DataFrame({
    "column_name": schema_df.columns,
    "data_type": schema_df.dtypes.astype(str).values,
    "missing_count_sample": schema_df.isnull().sum().values,
    "missing_percentage_sample": (
        schema_df.isnull().mean().values * 100
    ).round(2),
    "unique_values_sample": [
        schema_df[col].nunique(dropna=True)
        for col in schema_df.columns
    ]
})

schema_report.head()

,column_name,data_type,missing_count_sample,missing_percentage_sample,unique_values_sample
0,client_id,int64,0,0.0,354
1,client_tag_name,object,541,54.1,2
2,client_tag_num,int64,0,0.0,3
3,campaign_id,int64,0,0.0,1000
4,campaign_test_id,int64,0,0.0,5


In [ ]:
schema_output = "/content/drive/MyDrive/AI_Email_Deliverability_Intelligence/data/processed/schema_report.csv"

schema_report.to_csv(
    schema_output,
    index=False
)

print("Schema report saved to:")
print(schema_output)

Schema report saved to:
/content/drive/MyDrive/AI_Email_Deliverability_Intelligence/data/processed/schema_report.csv


In [ ]:
import os

BASE_DIR = "/content/drive/MyDrive/AI_Email_Deliverability_Intelligence"

print(os.path.exists(BASE_DIR))
print(os.listdir(BASE_DIR))

True
['data', 'notebooks', 'models', 'src', 'reports']


In [ ]:
file_path = os.path.join(
    BASE_DIR,
    "data",
    "raw",
    "SendGuard900K.csv"
)

print("File exists:", os.path.exists(file_path))

File exists: True


In [ ]:
import pandas as pd
import os

file_path = "/content/drive/MyDrive/AI_Email_Deliverability_Intelligence/data/raw/SendGuard900K.csv"

# Read a sample only for datatype inspection
sample_df = pd.read_csv(
    file_path,
    nrows=1000,
    low_memory=False
)

# --------------------------------------------------
# 1. Get complete missing-value counts
# --------------------------------------------------

missing_counts = pd.Series(0, index=sample_df.columns, dtype="int64")

for chunk in pd.read_csv(
    file_path,
    chunksize=50000,
    low_memory=False
):
    missing_counts = missing_counts.add(
        chunk.isnull().sum(),
        fill_value=0
    )

missing_counts = missing_counts.astype(int)

total_rows = 908229

# --------------------------------------------------
# 2. Function to classify columns
# --------------------------------------------------

def classify_column(col):

    if col in [
        "client_id",
        "client_tag_name",
        "client_tag_num",
        "campaign_id",
        "campaign_test_id"
    ]:
        return "Identity / Client"

    elif col == "campaign_sent_time":
        return "Timestamp"

    elif col.startswith("message_"):
        return "Message Characteristics"

    elif col in [
        "campaign_subscribers_count",
        "campaign_phantom_bounce_count",
        "campaign_subscribers_blocked_count"
    ]:
        return "Campaign Volume / Audience"

    elif "soft_bounced" in col or "hard_bounced" in col:
        return "Bounce / Delivery"

    elif "complaint" in col or "phishing" in col:
        return "Negative Deliverability Signals"

    elif "moderation" in col:
        return "Moderation"

    elif "landing_page" in col:
        return "Campaign Features"

    elif col in [
        "campaign_test_part",
        "campaign_test_type"
    ]:
        return "Campaign Testing"

    elif any(
        col.endswith("_" + str(x))
        for x in [0, 30, 60, 180, 360, 720, 1440, 2880, 4320]
    ):
        return "Historical / Time-Window Metrics"

    elif col.startswith("domain_in_links_"):
        return "Linked Domain Intelligence"

    elif col in [
        "registrar_domain_number",
        "domain_number"
    ]:
        return "Domain Counts"

    else:
        return "Other"

# --------------------------------------------------
# 3. Create inventory
# --------------------------------------------------

column_inventory = pd.DataFrame({
    "column_name": sample_df.columns,
    "data_type": [
        str(sample_df[col].dtype)
        for col in sample_df.columns
    ],
    "missing_count": [
        missing_counts[col]
        for col in sample_df.columns
    ]
})

column_inventory["missing_percentage"] = (
    column_inventory["missing_count"] / total_rows * 100
).round(2)

column_inventory["column_group"] = (
    column_inventory["column_name"]
    .apply(classify_column)
)

# --------------------------------------------------
# 4. Display
# --------------------------------------------------

print("Total columns:", len(column_inventory))

column_inventory

Total columns: 119


,column_name,data_type,missing_count,missing_percentage,column_group
0,client_id,int64,0,0.0,Identity / Client
1,client_tag_name,object,606654,66.8,Identity / Client
2,client_tag_num,int64,0,0.0,Identity / Client
3,campaign_id,int64,0,0.0,Identity / Client
4,campaign_test_id,int64,0,0.0,Identity / Client
...,...,...,...,...,...
114,domain_in_links_max_creation_time_diff,int64,0,0.0,Linked Domain Intelligence
115,domain_in_links_avg_creation_time_diff,int64,0,0.0,Linked Domain Intelligence
116,domain_in_links_most_used_creation_time_diff,int64,0,0.0,Linked Domain Intelligence
117,registrar_domain_number,int64,0,0.0,Domain Counts


In [ ]:
inventory_path = "/content/drive/MyDrive/AI_Email_Deliverability_Intelligence/data/processed/column_inventory.csv"

column_inventory.to_csv(
    inventory_path,
    index=False
)

print("Saved successfully:")
print(inventory_path)

Saved successfully:
/content/drive/MyDrive/AI_Email_Deliverability_Intelligence/data/processed/column_inventory.csv


In [ ]:
print(
    column_inventory["column_group"]
    .value_counts()
)

column_group
Historical / Time-Window Metrics    45
Bounce / Delivery                   20
Message Characteristics             13
Negative Deliverability Signals     11
Linked Domain Intelligence           8
Other                                6
Identity / Client                    5
Campaign Volume / Audience           3
Campaign Testing                     2
Domain Counts                        2
Moderation                           2
Timestamp                            1
Campaign Features                    1
Name: count, dtype: int64


In [ ]:
import pandas as pd
import numpy as np
import os

file_path = "/content/drive/MyDrive/AI_Email_Deliverability_Intelligence/data/raw/SendGuard900K.csv"

processed_dir = "/content/drive/MyDrive/AI_Email_Deliverability_Intelligence/data/processed"

os.makedirs(processed_dir, exist_ok=True)

df = pd.read_csv(
    file_path,
    low_memory=False
)

print("Original shape:", df.shape)

# Convert timestamp
df["campaign_sent_time"] = pd.to_datetime(
    df["campaign_sent_time"],
    errors="coerce",
    utc=True
)

# Normalize the mixed-type test columns
df["campaign_test_part"] = (
    df["campaign_test_part"]
    .astype(str)
    .replace("nan", "0")
)

df["campaign_test_type"] = (
    df["campaign_test_type"]
    .astype(str)
    .replace("nan", "0")
)

print("\nData types corrected.")
print(df[[
    "campaign_sent_time",
    "campaign_test_part",
    "campaign_test_type"
]].dtypes)

Original shape: (908229, 119)

Data types corrected.
campaign_sent_time    datetime64[ns, UTC]
campaign_test_part                 object
campaign_test_type                 object
dtype: object


In [ ]:
before = len(df)

exact_duplicate_count = df.duplicated().sum()

df = df.drop_duplicates().copy()

after = len(df)

print("Rows before:", before)
print("Exact duplicates:", exact_duplicate_count)
print("Rows after:", after)

Rows before: 908229
Exact duplicates: 3
Rows after: 908226


In [ ]:
sent = df["campaign_subscribers_count"].replace(0, np.nan)

df["delivery_rate"] = (
    (sent - df["campaign_soft_bounced_count"]
          - df["campaign_hard_bounced_count"])
    / sent
) * 100

df["soft_bounce_rate"] = (
    df["campaign_soft_bounced_count"] / sent
) * 100

df["hard_bounce_rate"] = (
    df["campaign_hard_bounced_count"] / sent
) * 100

df["complaint_rate"] = (
    df["campaign_complaint_count"] / sent
) * 100

df["phishing_rate"] = (
    df["campaign_phishing_count"] / sent
) * 100

df["open_rate"] = (
    df["campaign_unique_opens_count"] / sent
) * 100

df["click_rate"] = (
    df["campaign_unique_clicks_count"] / sent
) * 100

In [ ]:
df["sent_date"] = df["campaign_sent_time"].dt.date
df["sent_hour"] = df["campaign_sent_time"].dt.hour
df["sent_day_of_week"] = df["campaign_sent_time"].dt.dayofweek
df["sent_day_name"] = df["campaign_sent_time"].dt.day_name()
df["sent_month"] = df["campaign_sent_time"].dt.month
df["sent_year"] = df["campaign_sent_time"].dt.year

In [ ]:
metric_columns = [
    "delivery_rate",
    "soft_bounce_rate",
    "hard_bounce_rate",
    "complaint_rate",
    "phishing_rate",
    "open_rate",
    "click_rate"
]

print(
    df[metric_columns]
    .describe()
    .T
)

                     count       mean        std  min        25%        50%  \
delivery_rate     870013.0  98.449476   5.379653  0.0  99.197531  99.868178   
soft_bounce_rate  870013.0   0.457682   2.951374  0.0   0.000000   0.000000   
hard_bounce_rate  870013.0   1.092842   4.138216  0.0   0.000000   0.063251   
complaint_rate    870013.0   0.000041   0.003119  0.0   0.000000   0.000000   
phishing_rate     870013.0   0.000322   0.040501  0.0   0.000000   0.000000   
open_rate         870013.0  25.374643  21.367359  0.0  10.759494  20.347395   
click_rate        870013.0   4.160757  11.940212  0.0   0.000000   0.602410   

                         75%         max  
delivery_rate     100.000000  100.000000  
soft_bounce_rate    0.133741  100.000000  
hard_bounce_rate    0.482033  100.000000  
complaint_rate      0.000000    1.190476  
phishing_rate       0.000000   33.333333  
open_rate          33.333333  100.000000  
click_rate          2.565344  100.000000  


In [ ]:
processed_path = os.path.join(
    processed_dir,
    "sendguard_base_processed.parquet"
)

df.to_parquet(
    processed_path,
    index=False
)

print("Processed dataset saved:")
print(processed_path)

Processed dataset saved:
/content/drive/MyDrive/AI_Email_Deliverability_Intelligence/data/processed/sendguard_base_processed.parquet


In [ ]:
print("Total rows:", len(df))

print(
    "Zero subscriber campaigns:",
    (df["campaign_subscribers_count"] == 0).sum()
)

print(
    "Missing subscriber values:",
    df["campaign_subscribers_count"].isna().sum()
)

Total rows: 908226
Zero subscriber campaigns: 38213
Missing subscriber values: 0


In [ ]:
zero_sent = df[df["campaign_subscribers_count"] == 0]

print("Zero-subscriber rows:", len(zero_sent))

print(
    zero_sent[
        [
            "campaign_id",
            "client_id",
            "campaign_sent_time",
            "campaign_subscribers_count",
            "campaign_opens_count",
            "campaign_clicks_count",
            "campaign_soft_bounced_count",
            "campaign_hard_bounced_count",
            "campaign_complaint_count",
            "campaign_phishing_count",
            "campaign_moderation"
        ]
    ].head(20)
)

Zero-subscriber rows: 38213
     campaign_id  client_id        campaign_sent_time  \
48       5995688     424818 2024-04-01 05:24:04+00:00   
53       5995693     424818 2024-04-01 05:29:02+00:00   
54       5995694     424818 2024-04-01 05:30:02+00:00   
56       5995696     424818 2024-04-01 05:32:02+00:00   
61       5995701     424818 2024-04-01 05:36:05+00:00   
65       5995705     424818 2024-04-01 05:39:03+00:00   
152      5995779     424818 2024-04-01 06:16:01+00:00   
153      5995780     424818 2024-04-01 06:17:01+00:00   
156      5995782     424818 2024-04-01 06:19:01+00:00   
158      5995785     424818 2024-04-01 06:21:01+00:00   
161      5995789     424818 2024-04-01 06:24:02+00:00   
162      5995790     424818 2024-04-01 06:26:03+00:00   
164      5995791     424818 2024-04-01 06:28:02+00:00   
165      5995792     424818 2024-04-01 06:29:02+00:00   
167      5995793     424818 2024-04-01 06:30:03+00:00   
168      5995794     424818 2024-04-01 06:31:02+00:00   
169

In [ ]:
sent = df["campaign_subscribers_count"]

checks = pd.DataFrame({
    "soft_bounce_gt_sent":
        df["campaign_soft_bounced_count"] > sent,

    "hard_bounce_gt_sent":
        df["campaign_hard_bounced_count"] > sent,

    "opens_gt_sent":
        df["campaign_opens_count"] > sent,

    "unique_opens_gt_sent":
        df["campaign_unique_opens_count"] > sent,

    "clicks_gt_sent":
        df["campaign_clicks_count"] > sent,

    "unique_clicks_gt_sent":
        df["campaign_unique_clicks_count"] > sent,

    "complaints_gt_sent":
        df["campaign_complaint_count"] > sent,

    "phishing_gt_sent":
        df["campaign_phishing_count"] > sent
})

print(checks.sum())

soft_bounce_gt_sent          2
hard_bounce_gt_sent          4
opens_gt_sent            84708
unique_opens_gt_sent         7
clicks_gt_sent           17477
unique_clicks_gt_sent        6
complaints_gt_sent           0
phishing_gt_sent             0
dtype: int64


In [ ]:
unique_open_anomalies = df[
    df["campaign_unique_opens_count"]
    > df["campaign_subscribers_count"]
]

print("Rows:", len(unique_open_anomalies))

print(
    unique_open_anomalies[
        [
            "campaign_id",
            "client_id",
            "campaign_subscribers_count",
            "campaign_opens_count",
            "campaign_unique_opens_count",
            "campaign_clicks_count",
            "campaign_unique_clicks_count",
            "campaign_soft_bounced_count",
            "campaign_hard_bounced_count",
            "campaign_sent_time"
        ]
    ].to_string(index=False)
)

Rows: 7
 campaign_id  client_id  campaign_subscribers_count  campaign_opens_count  campaign_unique_opens_count  campaign_clicks_count  campaign_unique_clicks_count  campaign_soft_bounced_count  campaign_hard_bounced_count        campaign_sent_time
     5193095      95760                           0                    11                            7                      5                             2                            0                            3                       NaT
     5193487     399496                           0                     6                            2                      0                             0                            0                            0                       NaT
     5193592     436881                           0                   125                           64                      6                             5                            0                            2 2022-08-02 08:08:38+00:00
     5193379     186283         

In [ ]:
unique_click_anomalies = df[
    df["campaign_unique_clicks_count"]
    > df["campaign_subscribers_count"]
]

print("Rows:", len(unique_click_anomalies))

print(
    unique_click_anomalies[
        [
            "campaign_id",
            "client_id",
            "campaign_subscribers_count",
            "campaign_clicks_count",
            "campaign_unique_clicks_count",
            "campaign_sent_time"
        ]
    ].to_string(index=False)
)

Rows: 6
 campaign_id  client_id  campaign_subscribers_count  campaign_clicks_count  campaign_unique_clicks_count        campaign_sent_time
     5193095      95760                           0                      5                             2                       NaT
     5193592     436881                           0                      6                             5 2022-08-02 08:08:38+00:00
     5193379     186283                           0                     54                             1 2022-08-02 08:09:17+00:00
     5193535     102702                           0                     33                            19 2022-08-02 08:10:52+00:00
     5054447     183760                           0                     51                            36 2022-08-02 08:18:33+00:00
     5186047       6251                           0                     24                            20 2022-08-02 08:31:02+00:00


In [ ]:
bounce_anomalies = df[
    (
        df["campaign_soft_bounced_count"]
        > df["campaign_subscribers_count"]
    )
    |
    (
        df["campaign_hard_bounced_count"]
        > df["campaign_subscribers_count"]
    )
]

print("Rows:", len(bounce_anomalies))

print(
    bounce_anomalies[
        [
            "campaign_id",
            "client_id",
            "campaign_subscribers_count",
            "campaign_phantom_bounce_count",
            "campaign_subscribers_blocked_count",
            "campaign_soft_bounced_count",
            "campaign_hard_bounced_count",
            "campaign_complaint_count",
            "campaign_phishing_count",
            "campaign_sent_time"
        ]
    ].to_string(index=False)
)

Rows: 4
 campaign_id  client_id  campaign_subscribers_count  campaign_phantom_bounce_count  campaign_subscribers_blocked_count  campaign_soft_bounced_count  campaign_hard_bounced_count  campaign_complaint_count  campaign_phishing_count        campaign_sent_time
     5193095      95760                           0                              0                                   0                            0                            3                         0                        0                       NaT
     5193592     436881                           0                              0                                   0                            0                            2                         0                        0 2022-08-02 08:08:38+00:00
     5054447     183760                           0                              0                                   0                           33                          202                         0                        0 20

In [ ]:
# -----------------------------------------
# ANOMALY SUMMARY
# -----------------------------------------

anomaly_summary = {
    "unique_opens_gt_sent": (
        df["campaign_unique_opens_count"]
        > df["campaign_subscribers_count"]
    ).sum(),

    "unique_clicks_gt_sent": (
        df["campaign_unique_clicks_count"]
        > df["campaign_subscribers_count"]
    ).sum(),

    "soft_bounce_gt_sent": (
        df["campaign_soft_bounced_count"]
        > df["campaign_subscribers_count"]
    ).sum(),

    "hard_bounce_gt_sent": (
        df["campaign_hard_bounced_count"]
        > df["campaign_subscribers_count"]
    ).sum()
}

print("ANOMALY COUNTS")
print("-" * 40)

for key, value in anomaly_summary.items():
    print(f"{key}: {value}")


# -----------------------------------------
# SHOW ALL ANOMALOUS RECORDS
# -----------------------------------------

anomaly_mask = (
    (df["campaign_unique_opens_count"] > df["campaign_subscribers_count"]) |
    (df["campaign_unique_clicks_count"] > df["campaign_subscribers_count"]) |
    (df["campaign_soft_bounced_count"] > df["campaign_subscribers_count"]) |
    (df["campaign_hard_bounced_count"] > df["campaign_subscribers_count"])
)

anomalies = df.loc[
    anomaly_mask,
    [
        "campaign_id",
        "client_id",
        "campaign_subscribers_count",
        "campaign_opens_count",
        "campaign_unique_opens_count",
        "campaign_clicks_count",
        "campaign_unique_clicks_count",
        "campaign_soft_bounced_count",
        "campaign_hard_bounced_count",
        "campaign_complaint_count",
        "campaign_phishing_count",
        "campaign_sent_time"
    ]
]

print("\nANOMALOUS RECORDS")
print("-" * 40)

print(anomalies.to_string(index=False))

ANOMALY COUNTS
----------------------------------------
unique_opens_gt_sent: 7
unique_clicks_gt_sent: 6
soft_bounce_gt_sent: 2
hard_bounce_gt_sent: 4

ANOMALOUS RECORDS
----------------------------------------
 campaign_id  client_id  campaign_subscribers_count  campaign_opens_count  campaign_unique_opens_count  campaign_clicks_count  campaign_unique_clicks_count  campaign_soft_bounced_count  campaign_hard_bounced_count  campaign_complaint_count  campaign_phishing_count        campaign_sent_time
     5193095      95760                           0                    11                            7                      5                             2                            0                            3                         0                        0                       NaT
     5193487     399496                           0                     6                            2                      0                             0                            0                       

In [ ]:
print(
    anomalies["campaign_subscribers_count"]
    .value_counts(dropna=False)
)

campaign_subscribers_count
0    7
Name: count, dtype: int64


In [ ]:
print(
    anomalies[
        "campaign_subscribers_count"
    ].eq(0).all()
)

True


In [ ]:
zero_subscriber = df[
    df["campaign_subscribers_count"] == 0
]

print("Zero-subscriber campaigns:", len(zero_subscriber))

activity_columns = [
    "campaign_opens_count",
    "campaign_unique_opens_count",
    "campaign_clicks_count",
    "campaign_unique_clicks_count",
    "campaign_soft_bounced_count",
    "campaign_hard_bounced_count",
    "campaign_complaint_count",
    "campaign_phishing_count"
]

has_activity = (
    zero_subscriber[activity_columns]
    .sum(axis=1) > 0
)

print(
    "Zero-subscriber campaigns with recorded activity:",
    has_activity.sum()
)

print(
    "Percentage:",
    round(has_activity.mean() * 100, 2),
    "%"
)

Zero-subscriber campaigns: 38213
Zero-subscriber campaigns with recorded activity: 7
Percentage: 0.02 %


In [ ]:
activity_columns = [
    "campaign_opens_count",
    "campaign_unique_opens_count",
    "campaign_clicks_count",
    "campaign_unique_clicks_count",
    "campaign_soft_bounced_count",
    "campaign_hard_bounced_count",
    "campaign_complaint_count",
    "campaign_phishing_count"
]

df["has_valid_audience"] = (
    df["campaign_subscribers_count"] > 0
)

df["zero_audience_with_activity"] = (
    (df["campaign_subscribers_count"] == 0) &
    (df[activity_columns].sum(axis=1) > 0)
)

print(
    "Invalid/zero audience:",
    (~df["has_valid_audience"]).sum()
)

print(
    "Zero audience with activity:",
    df["zero_audience_with_activity"].sum()
)

Invalid/zero audience: 38213
Zero audience with activity: 7


In [ ]:
processed_path = "/content/drive/MyDrive/AI_Email_Deliverability_Intelligence/data/processed/sendguard_base_processed.parquet"

df.to_parquet(
    processed_path,
    index=False
)

print("Updated processed dataset saved.")
print(processed_path)

Updated processed dataset saved.
/content/drive/MyDrive/AI_Email_Deliverability_Intelligence/data/processed/sendguard_base_processed.parquet


In [ ]:
# ============================================
# NUMERICAL DATA QUALITY AUDIT
# ============================================

numeric_columns = df.select_dtypes(include="number").columns

print("Total numeric columns:", len(numeric_columns))

# --------------------------------------------
# 1. Negative values
# --------------------------------------------

negative_counts = (df[numeric_columns] < 0).sum()

negative_summary = (
    negative_counts[negative_counts > 0]
    .sort_values(ascending=False)
)

print("\nColumns containing negative values:")
print(negative_summary)

# --------------------------------------------
# 2. Constant columns
# --------------------------------------------

unique_counts = df[numeric_columns].nunique(dropna=False)

constant_columns = unique_counts[
    unique_counts <= 1
].sort_values()

print("\nConstant numeric columns:")
print(constant_columns)

# --------------------------------------------
# 3. All-zero columns
# --------------------------------------------

zero_only_columns = []

for col in numeric_columns:
    if (df[col].fillna(0) == 0).all():
        zero_only_columns.append(col)

print("\nAll-zero numeric columns:")
print(zero_only_columns)

# --------------------------------------------
# 4. Very high maximum values
# --------------------------------------------

numeric_summary = df[numeric_columns].describe().T

numeric_summary["missing_count"] = (
    df[numeric_columns].isna().sum()
)

numeric_summary["missing_percentage"] = (
    df[numeric_columns].isna().mean() * 100
).round(2)

print("\nNumeric summary:")
display(numeric_summary)

Total numeric columns: 122

Columns containing negative values:
domain_in_links_most_used_creation_time_diff    437221
domain_in_links_min_creation_time_diff          182273
domain_in_links_max_creation_time_diff          182273
domain_in_links_avg_creation_time_diff          182273
dtype: int64

Constant numeric columns:
Series([], dtype: int64)

All-zero numeric columns:
[]

Numeric summary:


,count,mean,std,min,25%,50%,75%,max,missing_count,missing_percentage
client_id,908226.0,3.238895e+05,189263.247578,17.0,110831.00,4.248180e+05,4.768970e+05,526583.0,0,0.00
client_tag_num,908226.0,3.661457e-01,0.566700,0.0,0.00,0.000000e+00,1.000000e+00,3.0,0,0.00
campaign_id,908226.0,5.648251e+06,390178.788558,3159231.0,5345795.25,5.686076e+06,5.986218e+06,6260790.0,0,0.00
campaign_test_id,908226.0,2.086980e+03,13329.581014,0.0,0.00,0.000000e+00,0.000000e+00,91816.0,0,0.00
message_content_size,908226.0,3.940187e+04,37684.604646,0.0,17373.00,3.293200e+04,4.949600e+04,523923.0,0,0.00
...,...,...,...,...,...,...,...,...,...,...
click_rate,870013.0,4.160757e+00,11.940212,0.0,0.00,6.024096e-01,2.565344e+00,100.0,38213,4.21
sent_hour,890665.0,9.369099e+00,3.493859,0.0,7.00,9.000000e+00,1.200000e+01,23.0,17561,1.93
sent_day_of_week,890665.0,2.238070e+00,1.638078,0.0,1.00,2.000000e+00,4.000000e+00,6.0,17561,1.93
sent_month,890665.0,6.484382e+00,3.427462,1.0,3.00,6.000000e+00,1.000000e+01,12.0,17561,1.93


In [ ]:
# ============================================
# TIMESTAMP QUALITY AUDIT
# ============================================

raw_dates = pd.read_csv(
    file_path,
    usecols=["campaign_sent_time"],
    dtype={"campaign_sent_time": "string"},
    low_memory=False
)

parsed_dates = pd.to_datetime(
    raw_dates["campaign_sent_time"],
    errors="coerce",
    utc=True
)

invalid_timestamp_mask = (
    raw_dates["campaign_sent_time"].notna() &
    parsed_dates.isna()
)

print("Total timestamp records:", len(raw_dates))
print("Valid timestamps:", parsed_dates.notna().sum())
print("Invalid timestamps:", invalid_timestamp_mask.sum())
print(
    "Invalid percentage:",
    round(invalid_timestamp_mask.mean() * 100, 2),
    "%"
)

print("\nExamples of invalid timestamp values:")
print(
    raw_dates.loc[
        invalid_timestamp_mask,
        "campaign_sent_time"
    ].value_counts().head(20)
)

Total timestamp records: 908229
Valid timestamps: 890668
Invalid timestamps: 17561
Invalid percentage: 1.93 %

Examples of invalid timestamp values:
campaign_sent_time
2023-02-14T08:12Z         4
2023-11-28T08:01Z         4
2023-01-25T07:01Z         3
2023-07-17T06:01+01:00    3
2023-06-12T06:01+01:00    3
2023-02-23T08:02Z         3
2024-03-14T07:01Z         3
2024-04-25T06:01+01:00    3
2023-10-05T07:24+01:00    3
2023-09-15T08:01+01:00    3
2023-04-24T08:01+01:00    3
2023-10-17T10:01+01:00    3
2024-04-30T10:49+01:00    3
2024-01-02T08:01Z         3
2023-05-09T07:01+01:00    3
2022-05-26T07:01+01:00    3
2024-06-06T05:01+01:00    3
2024-10-21T07:56+01:00    3
2024-05-21T06:01+01:00    3
2023-01-09T12:19Z         2
Name: count, dtype: Int64


In [ ]:
# Check completely missing timestamp values
missing_timestamp_mask = raw_dates["campaign_sent_time"].isna()

print("Missing timestamp values:", missing_timestamp_mask.sum())

Missing timestamp values: 0


In [ ]:
print("Negative-value columns:")
print(negative_summary)

print("\nConstant columns:")
print(constant_columns)

print("\nAll-zero columns:")
print(zero_only_columns)

Negative-value columns:
domain_in_links_most_used_creation_time_diff    437221
domain_in_links_min_creation_time_diff          182273
domain_in_links_max_creation_time_diff          182273
domain_in_links_avg_creation_time_diff          182273
dtype: int64

Constant columns:
Series([], dtype: int64)

All-zero columns:
[]


In [ ]:
# ============================================
# NEGATIVE DOMAIN AGE DIFFERENCE AUDIT
# ============================================

negative_domain_cols = [
    "domain_in_links_min_creation_time_diff",
    "domain_in_links_max_creation_time_diff",
    "domain_in_links_avg_creation_time_diff",
    "domain_in_links_most_used_creation_time_diff"
]

negative_mask = False

for col in negative_domain_cols:
    negative_mask = negative_mask | (df[col] < 0)

negative_domain_data = df.loc[
    negative_mask,
    [
        "campaign_id",
        "client_id",
        "campaign_sent_time",
        "domain_in_links_min_creation_time",
        "domain_in_links_max_creation_time",
        "domain_in_links_avg_creation_time",
        "domain_in_links_most_used_creation_time",
        "domain_in_links_min_creation_time_diff",
        "domain_in_links_max_creation_time_diff",
        "domain_in_links_avg_creation_time_diff",
        "domain_in_links_most_used_creation_time_diff"
    ]
]

print("Rows with at least one negative domain-age difference:",
      len(negative_domain_data))

display(negative_domain_data.head(20))

Rows with at least one negative domain-age difference: 437221


,campaign_id,client_id,campaign_sent_time,domain_in_links_min_creation_time,domain_in_links_max_creation_time,domain_in_links_avg_creation_time,domain_in_links_most_used_creation_time,domain_in_links_min_creation_time_diff,domain_in_links_max_creation_time_diff,domain_in_links_avg_creation_time_diff,domain_in_links_most_used_creation_time_diff
0,5995662,171728,2024-04-01 02:00:07+00:00,2004-06-04,2004-06-04,2004-06-04,NaN,7241,7241,7241,-1
2,5995663,171728,2024-04-01 02:00:08+00:00,2004-06-04,2004-06-04,2004-06-04,NaN,7241,7241,7241,-1
5,5993861,481573,2024-04-01 03:00:11+00:00,1997-03-29,2004-06-04,2000-10-31,NaN,9865,7241,8553,-1
7,5994425,44833,2024-04-01 03:02:07+00:00,1997-03-29,2005-02-15,2002-04-07,NaN,9865,6985,8030,-1
9,5992378,480559,2024-04-01 04:00:08+00:00,2004-09-21,2004-09-21,2004-09-21,NaN,7132,7132,7132,-1
10,5992737,510945,2024-04-01 04:00:17+00:00,1997-03-29,2009-07-24,2003-09-29,NaN,9865,5365,7490,-1
12,5995653,505583,2024-04-01 04:00:43+00:00,1997-03-29,2005-02-15,2002-04-07,NaN,9865,6985,8030,-1
18,5994954,458854,2024-04-01 04:15:03+00:00,NaN,NaN,NaN,NaN,-1,-1,-1,-1
20,5959439,454776,2024-04-01 04:30:08+00:00,1997-03-29,2004-06-04,2000-10-31,NaN,9865,7241,8553,-1
21,5991501,13317,2024-04-01 04:30:54+00:00,1997-03-29,2005-02-15,2002-04-07,NaN,9865,6985,8030,-1


In [ ]:
for col in negative_domain_cols:
    values = df.loc[df[col] < 0, col]

    print("\n", col)
    print("Negative count:", len(values))
    print("Minimum:", values.min())
    print("Maximum:", values.max())


 domain_in_links_min_creation_time_diff
Negative count: 182273
Minimum: -1
Maximum: -1

 domain_in_links_max_creation_time_diff
Negative count: 182273
Minimum: -1
Maximum: -1

 domain_in_links_avg_creation_time_diff
Negative count: 182273
Minimum: -1
Maximum: -1

 domain_in_links_most_used_creation_time_diff
Negative count: 437221
Minimum: -1
Maximum: -1


In [ ]:
negative_domain_cols = [
    "domain_in_links_min_creation_time_diff",
    "domain_in_links_max_creation_time_diff",
    "domain_in_links_avg_creation_time_diff",
    "domain_in_links_most_used_creation_time_diff"
]

df[negative_domain_cols] = (
    df[negative_domain_cols]
    .replace(-1, np.nan)
)

In [ ]:
for col in negative_domain_cols:
    print(
        col,
        "negative values:",
        (df[col] < 0).sum(),
        "| missing:",
        df[col].isna().sum()
    )

domain_in_links_min_creation_time_diff negative values: 0 | missing: 182273
domain_in_links_max_creation_time_diff negative values: 0 | missing: 182273
domain_in_links_avg_creation_time_diff negative values: 0 | missing: 182273
domain_in_links_most_used_creation_time_diff negative values: 0 | missing: 437221


In [ ]:
# ============================================
# TIMESTAMP QUALITY AUDIT
# ============================================

raw_dates = pd.read_csv(
    file_path,
    usecols=["campaign_sent_time"],
    dtype={"campaign_sent_time": "string"},
    low_memory=False
)

parsed_dates = pd.to_datetime(
    raw_dates["campaign_sent_time"],
    errors="coerce",
    utc=True
)

invalid_timestamp_mask = (
    raw_dates["campaign_sent_time"].notna() &
    parsed_dates.isna()
)

missing_timestamp_mask = (
    raw_dates["campaign_sent_time"].isna()
)

print("Total records:", len(raw_dates))
print("Valid timestamps:", parsed_dates.notna().sum())
print("Invalid timestamps:", invalid_timestamp_mask.sum())
print("Missing timestamps:", missing_timestamp_mask.sum())

print(
    "Invalid percentage:",
    round(invalid_timestamp_mask.mean() * 100, 2),
    "%"
)

print("\nExamples of invalid timestamp values:")
print(
    raw_dates.loc[
        invalid_timestamp_mask,
        "campaign_sent_time"
    ].value_counts().head(20)
)

Total records: 908229
Valid timestamps: 890668
Invalid timestamps: 17561
Missing timestamps: 0
Invalid percentage: 1.93 %

Examples of invalid timestamp values:
campaign_sent_time
2023-02-14T08:12Z         4
2023-11-28T08:01Z         4
2023-01-25T07:01Z         3
2023-07-17T06:01+01:00    3
2023-06-12T06:01+01:00    3
2023-02-23T08:02Z         3
2024-03-14T07:01Z         3
2024-04-25T06:01+01:00    3
2023-10-05T07:24+01:00    3
2023-09-15T08:01+01:00    3
2023-04-24T08:01+01:00    3
2023-10-17T10:01+01:00    3
2024-04-30T10:49+01:00    3
2024-01-02T08:01Z         3
2023-05-09T07:01+01:00    3
2022-05-26T07:01+01:00    3
2024-06-06T05:01+01:00    3
2024-10-21T07:56+01:00    3
2024-05-21T06:01+01:00    3
2023-01-09T12:19Z         2
Name: count, dtype: Int64


In [ ]:
print("\nExamples of completely missing timestamps:")
print(
    raw_dates.loc[
        missing_timestamp_mask,
        "campaign_sent_time"
    ].head(20)
)


Examples of completely missing timestamps:
Series([], Name: campaign_sent_time, dtype: string)


In [ ]:
print("Invalid timestamp examples:")

print(
    raw_dates.loc[
        invalid_timestamp_mask,
        "campaign_sent_time"
    ].value_counts().head(30)
)

Invalid timestamp examples:
campaign_sent_time
2023-02-14T08:12Z         4
2023-11-28T08:01Z         4
2023-01-25T07:01Z         3
2023-07-17T06:01+01:00    3
2023-06-12T06:01+01:00    3
2023-02-23T08:02Z         3
2024-03-14T07:01Z         3
2024-04-25T06:01+01:00    3
2023-10-05T07:24+01:00    3
2023-09-15T08:01+01:00    3
2023-04-24T08:01+01:00    3
2023-10-17T10:01+01:00    3
2024-04-30T10:49+01:00    3
2024-01-02T08:01Z         3
2023-05-09T07:01+01:00    3
2022-05-26T07:01+01:00    3
2024-06-06T05:01+01:00    3
2024-10-21T07:56+01:00    3
2024-05-21T06:01+01:00    3
2023-01-09T12:19Z         2
2024-10-10T06:28+01:00    2
2023-04-20T06:29+01:00    2
2022-04-22T09:36+01:00    2
2024-08-21T07:09+01:00    2
2024-09-17T07:32+01:00    2
2024-03-24T18:01Z         2
2024-03-25T07:18Z         2
2022-09-19T07:02+01:00    2
2023-03-27T08:21+01:00    2
2023-10-18T10:02+01:00    2
Name: count, dtype: Int64


In [ ]:
invalid_values = raw_dates.loc[
    invalid_timestamp_mask,
    "campaign_sent_time"
]

print("Number of unique invalid timestamp values:",
      invalid_values.nunique())

print("\nFirst 30 invalid values:")
print(invalid_values.head(30).to_string(index=False))

Number of unique invalid timestamp values: 17184

First 30 invalid values:
2024-04-01T07:01+01:00
2024-04-01T07:02+01:00
2024-04-01T07:09+01:00
2024-04-01T07:11+01:00
2024-04-01T08:00+01:00
2024-04-02T04:56+01:00
2024-04-02T05:54+01:00
2024-04-02T06:01+01:00
2024-04-02T06:42+01:00
2024-04-02T06:43+01:00
2024-04-02T06:44+01:00
2024-04-02T06:46+01:00
2024-04-02T07:04+01:00
2024-04-02T07:06+01:00
2024-04-02T07:07+01:00
2024-04-02T07:55+01:00
2024-04-02T08:00+01:00
2024-04-02T08:01+01:00
2024-04-02T08:01+01:00
2024-04-02T09:34+01:00
2024-04-02T09:43+01:00
2024-04-02T09:54+01:00
2024-04-02T11:05+01:00
2024-04-02T13:09+01:00
2024-04-02T14:16+01:00
2024-04-02T14:20+01:00
2024-04-02T15:01+01:00
2024-04-03T05:02+01:00
2024-04-03T05:10+01:00
2024-04-03T06:18+01:00


In [ ]:
# ============================================
# REBUILD TIMESTAMP COLUMN CORRECTLY
# ============================================

import pandas as pd
import numpy as np

# Read the raw dataset again
df = pd.read_csv(
    file_path,
    low_memory=False
)

print("Raw shape:", df.shape)

# Remove exact duplicate rows
before = len(df)

df = df.drop_duplicates().copy()

print("Exact duplicates removed:", before - len(df))
print("Shape after deduplication:", df.shape)

# Keep original timestamp temporarily
original_timestamp = df["campaign_sent_time"].copy()

# Correctly parse mixed timestamp formats
df["campaign_sent_time"] = pd.to_datetime(
    original_timestamp,
    format="mixed",
    errors="coerce",
    utc=True
)

print("\nTimestamp parsing completed.")

print(
    "Invalid timestamps:",
    df["campaign_sent_time"].isna().sum()
)

print(
    "Valid timestamps:",
    df["campaign_sent_time"].notna().sum()
)

Raw shape: (908229, 119)
Exact duplicates removed: 3
Shape after deduplication: (908226, 119)

Timestamp parsing completed.
Invalid timestamps: 0
Valid timestamps: 908226


In [ ]:
df["sent_date"] = df["campaign_sent_time"].dt.date
df["sent_hour"] = df["campaign_sent_time"].dt.hour
df["sent_day_of_week"] = df["campaign_sent_time"].dt.dayofweek
df["sent_day_name"] = df["campaign_sent_time"].dt.day_name()
df["sent_month"] = df["campaign_sent_time"].dt.month
df["sent_year"] = df["campaign_sent_time"].dt.year

In [ ]:
print("Missing sent_hour:", df["sent_hour"].isna().sum())
print("Missing sent_day_of_week:", df["sent_day_of_week"].isna().sum())
print("Missing sent_month:", df["sent_month"].isna().sum())
print("Missing sent_year:", df["sent_year"].isna().sum())

Missing sent_hour: 0
Missing sent_day_of_week: 0
Missing sent_month: 0
Missing sent_year: 0


In [ ]:
print("Earliest:", df["campaign_sent_time"].min())
print("Latest:", df["campaign_sent_time"].max())

Earliest: 2022-01-01 05:00:05+00:00
Latest: 2024-12-31 23:10:08+00:00


In [ ]:
# ============================================
# CLEAN DOMAIN AGE SENTINEL VALUES
# ============================================

negative_domain_cols = [
    "domain_in_links_min_creation_time_diff",
    "domain_in_links_max_creation_time_diff",
    "domain_in_links_avg_creation_time_diff",
    "domain_in_links_most_used_creation_time_diff"
]

df[negative_domain_cols] = (
    df[negative_domain_cols]
    .replace(-1, np.nan)
)

print("Domain sentinel values converted to NaN.")

for col in negative_domain_cols:
    print(
        col,
        "negative:",
        (df[col] < 0).sum(),
        "| missing:",
        df[col].isna().sum()
    )

Domain sentinel values converted to NaN.
domain_in_links_min_creation_time_diff negative: 0 | missing: 182273
domain_in_links_max_creation_time_diff negative: 0 | missing: 182273
domain_in_links_avg_creation_time_diff negative: 0 | missing: 182273
domain_in_links_most_used_creation_time_diff negative: 0 | missing: 437221


In [ ]:
# ============================================
# AUDIENCE QUALITY FLAGS
# ============================================

activity_columns = [
    "campaign_opens_count",
    "campaign_unique_opens_count",
    "campaign_clicks_count",
    "campaign_unique_clicks_count",
    "campaign_soft_bounced_count",
    "campaign_hard_bounced_count",
    "campaign_complaint_count",
    "campaign_phishing_count"
]

df["has_valid_audience"] = (
    df["campaign_subscribers_count"] > 0
)

df["zero_audience_with_activity"] = (
    (df["campaign_subscribers_count"] == 0) &
    (df[activity_columns].sum(axis=1) > 0)
)

print(
    "Zero/invalid audience:",
    (~df["has_valid_audience"]).sum()
)

print(
    "Zero audience with activity:",
    df["zero_audience_with_activity"].sum()
)

Zero/invalid audience: 38213
Zero audience with activity: 7


In [ ]:
# ============================================
# DERIVED CAMPAIGN METRICS
# ============================================

sent = df["campaign_subscribers_count"].replace(0, np.nan)

df["provisional_delivery_rate"] = (
    (
        sent
        - df["campaign_soft_bounced_count"]
        - df["campaign_hard_bounced_count"]
    ) / sent
) * 100

df["soft_bounce_rate"] = (
    df["campaign_soft_bounced_count"] / sent
) * 100

df["hard_bounce_rate"] = (
    df["campaign_hard_bounced_count"] / sent
) * 100

df["complaint_rate"] = (
    df["campaign_complaint_count"] / sent
) * 100

df["phishing_rate"] = (
    df["campaign_phishing_count"] / sent
) * 100

df["open_rate"] = (
    df["campaign_unique_opens_count"] / sent
) * 100

df["click_rate"] = (
    df["campaign_unique_clicks_count"] / sent
) * 100

In [ ]:
# ============================================
# TIME FEATURES
# ============================================

df["sent_date"] = df["campaign_sent_time"].dt.date
df["sent_hour"] = df["campaign_sent_time"].dt.hour
df["sent_day_of_week"] = df["campaign_sent_time"].dt.dayofweek
df["sent_day_name"] = df["campaign_sent_time"].dt.day_name()
df["sent_month"] = df["campaign_sent_time"].dt.month
df["sent_year"] = df["campaign_sent_time"].dt.year

In [ ]:
print("Missing sent_hour:",
      df["sent_hour"].isna().sum())

print("Missing sent_day_of_week:",
      df["sent_day_of_week"].isna().sum())

print("Missing sent_month:",
      df["sent_month"].isna().sum())

print("Missing sent_year:",
      df["sent_year"].isna().sum())

Missing sent_hour: 0
Missing sent_day_of_week: 0
Missing sent_month: 0
Missing sent_year: 0


In [ ]:
# ============================================
# COMPLETE MISSING-VALUE AUDIT
# ============================================

missing_report = pd.DataFrame({
    "column": df.columns,
    "missing_count": df.isna().sum().values,
    "missing_percentage": (
        df.isna().mean().values * 100
    ).round(2),
    "data_type": df.dtypes.astype(str).values
})

missing_report = (
    missing_report[
        missing_report["missing_count"] > 0
    ]
    .sort_values("missing_percentage", ascending=False)
    .reset_index(drop=True)
)

print("Columns with missing values:", len(missing_report))

display(missing_report)

Columns with missing values: 16


,column,missing_count,missing_percentage,data_type
0,client_tag_name,606651,66.80,object
1,domain_in_links_most_used_creation_time,437221,48.14,object
2,domain_in_links_most_used_creation_time_diff,437221,48.14,float64
3,domain_in_links_max_creation_time,182273,20.07,object
4,domain_in_links_avg_creation_time,182273,20.07,object
5,domain_in_links_min_creation_time_diff,182273,20.07,float64
6,domain_in_links_max_creation_time_diff,182273,20.07,float64
7,domain_in_links_min_creation_time,182273,20.07,object
8,domain_in_links_avg_creation_time_diff,182273,20.07,float64
9,provisional_delivery_rate,38213,4.21,float64


In [ ]:
print("Total missing cells:", df.isna().sum().sum())

Total missing cells: 2842222


In [ ]:
high_missing = missing_report[
    missing_report["missing_percentage"] >= 20
]

display(high_missing)

,column,missing_count,missing_percentage,data_type
0,client_tag_name,606651,66.80,object
1,domain_in_links_most_used_creation_time,437221,48.14,object
2,domain_in_links_most_used_creation_time_diff,437221,48.14,float64
3,domain_in_links_max_creation_time,182273,20.07,object
4,domain_in_links_avg_creation_time,182273,20.07,object
5,domain_in_links_min_creation_time_diff,182273,20.07,float64
6,domain_in_links_max_creation_time_diff,182273,20.07,float64
7,domain_in_links_min_creation_time,182273,20.07,object
8,domain_in_links_avg_creation_time_diff,182273,20.07,float64


In [ ]:
domain_date_cols = [
    "domain_in_links_min_creation_time",
    "domain_in_links_max_creation_time",
    "domain_in_links_avg_creation_time",
    "domain_in_links_most_used_creation_time"
]

domain_diff_cols = [
    "domain_in_links_min_creation_time_diff",
    "domain_in_links_max_creation_time_diff",
    "domain_in_links_avg_creation_time_diff",
    "domain_in_links_most_used_creation_time_diff"
]

domain_missing_check = pd.DataFrame({
    "creation_date_missing": df[domain_date_cols].isna().sum(),
    "difference_missing": df[domain_diff_cols].isna().sum()
})

display(domain_missing_check)

,creation_date_missing,difference_missing
domain_in_links_avg_creation_time,182273.0,NaN
domain_in_links_avg_creation_time_diff,NaN,182273.0
domain_in_links_max_creation_time,182273.0,NaN
domain_in_links_max_creation_time_diff,NaN,182273.0
domain_in_links_min_creation_time,182273.0,NaN
domain_in_links_min_creation_time_diff,NaN,182273.0
domain_in_links_most_used_creation_time,437221.0,NaN
domain_in_links_most_used_creation_time_diff,NaN,437221.0


In [ ]:
print(
    df["client_tag_name"]
    .value_counts(dropna=False)
)

client_tag_name
NaN            606651
Inne           280085
Chwilówka       12012
Dom mediowy      9478
Name: count, dtype: int64


In [ ]:
# ============================================
# CLIENT TAG MISSINGNESS CONSISTENCY
# ============================================

print(
    pd.crosstab(
        df["client_tag_num"],
        df["client_tag_name"].isna(),
        margins=True
    )
)

In [ ]:
# ============================================
# DOMAIN MISSINGNESS CONSISTENCY
# ============================================

domain_columns = [
    "domain_in_links_min_creation_time",
    "domain_in_links_max_creation_time",
    "domain_in_links_avg_creation_time",
    "domain_in_links_most_used_creation_time"
]

for col in domain_columns:
    print("\n", "=" * 60)
    print(col)

    missing_mask = df[col].isna()

    print(
        "Missing rows:",
        missing_mask.sum()
    )

    print(
        "Domain_number distribution for missing values:"
    )

    print(
        df.loc[
            missing_mask,
            "domain_number"
        ].value_counts().sort_index().head(20)
    )


domain_in_links_min_creation_time
Missing rows: 182273
Domain_number distribution for missing values:
domain_number
0      42398
1     105128
2      27789
3       3965
4       1958
5        794
6         84
7         82
8         17
9         17
10         5
11         5
12         4
13        25
14         1
15         1
Name: count, dtype: int64

domain_in_links_max_creation_time
Missing rows: 182273
Domain_number distribution for missing values:
domain_number
0      42398
1     105128
2      27789
3       3965
4       1958
5        794
6         84
7         82
8         17
9         17
10         5
11         5
12         4
13        25
14         1
15         1
Name: count, dtype: int64

domain_in_links_avg_creation_time
Missing rows: 182273
Domain_number distribution for missing values:
domain_number
0      42398
1     105128
2      27789
3       3965
4       1958
5        794
6         84
7         82
8         17
9         17
10         5
11         5
12         4
13        25

In [ ]:
# ============================================
# DOMAIN DATE MISSINGNESS PATTERN
# ============================================

display(
    df[domain_columns]
    .isna()
    .value_counts()
    .rename("row_count")
    .reset_index()
)

,domain_in_links_min_creation_time,domain_in_links_max_creation_time,domain_in_links_avg_creation_time,domain_in_links_most_used_creation_time,row_count
0,False,False,False,False,471005
1,False,False,False,True,254948
2,True,True,True,True,182273


In [ ]:
# ============================================
# DOMAIN INFORMATION AVAILABILITY FLAGS
# ============================================

df["has_domain_creation_data"] = (
    df[
        [
            "domain_in_links_min_creation_time",
            "domain_in_links_max_creation_time",
            "domain_in_links_avg_creation_time",
            "domain_in_links_most_used_creation_time"
        ]
    ]
    .notna()
    .any(axis=1)
)

df["has_most_used_domain_creation_data"] = (
    df["domain_in_links_most_used_creation_time"].notna()
)

print(
    "Rows with any domain creation information:",
    df["has_domain_creation_data"].sum()
)

print(
    "Rows with most-used domain creation information:",
    df["has_most_used_domain_creation_data"].sum()
)

Rows with any domain creation information: 725953
Rows with most-used domain creation information: 471005


In [ ]:
# ============================================
# DOMAIN CREATION DATE TYPES
# ============================================

domain_date_columns = [
    "domain_in_links_min_creation_time",
    "domain_in_links_max_creation_time",
    "domain_in_links_avg_creation_time",
    "domain_in_links_most_used_creation_time"
]

for col in domain_date_columns:
    df[col] = pd.to_datetime(
        df[col],
        errors="coerce",
        utc=True,
        format="mixed"
    )

print(df[domain_date_columns].dtypes)

domain_in_links_min_creation_time          datetime64[ns, UTC]
domain_in_links_max_creation_time          datetime64[ns, UTC]
domain_in_links_avg_creation_time          datetime64[ns, UTC]
domain_in_links_most_used_creation_time    datetime64[ns, UTC]
dtype: object


In [ ]:
# ============================================
# CLIENT TAG HANDLING
# ============================================

df["client_tag_name"] = (
    df["client_tag_name"]
    .fillna("No_Tag")
)

print(
    df["client_tag_name"]
    .value_counts(dropna=False)
)

client_tag_name
No_Tag         606651
Inne           280085
Chwilówka       12012
Dom mediowy      9478
Name: count, dtype: int64


In [ ]:
# ============================================
# POST-TREATMENT MISSING-VALUE CHECK
# ============================================

remaining_missing = (
    df.isna()
    .sum()
    .sort_values(ascending=False)
)

remaining_missing = remaining_missing[
    remaining_missing > 0
]

print("Columns still containing missing values:")
print(remaining_missing)

Columns still containing missing values:
domain_in_links_most_used_creation_time         437221
domain_in_links_most_used_creation_time_diff    437221
domain_in_links_avg_creation_time               182273
domain_in_links_max_creation_time               182273
domain_in_links_max_creation_time_diff          182273
domain_in_links_min_creation_time_diff          182273
domain_in_links_avg_creation_time_diff          182273
domain_in_links_min_creation_time               182273
complaint_rate                                   38213
open_rate                                        38213
soft_bounce_rate                                 38213
hard_bounce_rate                                 38213
provisional_delivery_rate                        38213
click_rate                                       38213
phishing_rate                                    38213
dtype: int64


In [ ]:
missing_treatment = pd.DataFrame({
    "column": [
        "client_tag_name",
        "domain_in_links_min_creation_time",
        "domain_in_links_max_creation_time",
        "domain_in_links_avg_creation_time",
        "domain_in_links_most_used_creation_time",
        "domain_in_links_min_creation_time_diff",
        "domain_in_links_max_creation_time_diff",
        "domain_in_links_avg_creation_time_diff",
        "domain_in_links_most_used_creation_time_diff"
    ],
    "treatment": [
        "Missing → No_Tag",
        "Preserve NaN",
        "Preserve NaN",
        "Preserve NaN",
        "Preserve NaN",
        "Preserve NaN",
        "Preserve NaN",
        "Preserve NaN",
        "Preserve NaN"
    ],
    "reason": [
        "Absence of client tag is meaningful",
        "Domain information unavailable",
        "Domain information unavailable",
        "Domain information unavailable",
        "Domain information unavailable",
        "Domain information unavailable",
        "Domain information unavailable",
        "Domain information unavailable",
        "Domain information unavailable"
    ]
})

display(missing_treatment)

,column,treatment,reason
0,client_tag_name,Missing → No_Tag,Absence of client tag is meaningful
1,domain_in_links_min_creation_time,Preserve NaN,Domain information unavailable
2,domain_in_links_max_creation_time,Preserve NaN,Domain information unavailable
3,domain_in_links_avg_creation_time,Preserve NaN,Domain information unavailable
4,domain_in_links_most_used_creation_time,Preserve NaN,Domain information unavailable
5,domain_in_links_min_creation_time_diff,Preserve NaN,Domain information unavailable
6,domain_in_links_max_creation_time_diff,Preserve NaN,Domain information unavailable
7,domain_in_links_avg_creation_time_diff,Preserve NaN,Domain information unavailable
8,domain_in_links_most_used_creation_time_diff,Preserve NaN,Domain information unavailable


In [ ]:
missing_treatment_path = (
    "/content/drive/MyDrive/"
    "AI_Email_Deliverability_Intelligence/"
    "reports/missing_value_treatment.csv"
)

missing_treatment.to_csv(
    missing_treatment_path,
    index=False
)

print("Saved:", missing_treatment_path)

Saved: /content/drive/MyDrive/AI_Email_Deliverability_Intelligence/reports/missing_value_treatment.csv


In [ ]:
# ============================================
# OUTLIER / DISTRIBUTION AUDIT
# ============================================

outlier_columns = [
    "message_content_size",
    "message_images_count",
    "message_embedded_images_count",
    "message_embedded_images_size",
    "message_embedded_files_count",
    "message_embedded_files_size",
    "message_links_count",
    "message_links_blocked_count",
    "campaign_subscribers_count",
    "campaign_phantom_bounce_count",
    "campaign_subscribers_blocked_count",
    "campaign_opens_count",
    "campaign_unique_opens_count",
    "campaign_clicks_count",
    "campaign_unique_clicks_count",
    "campaign_soft_bounced_count",
    "campaign_hard_bounced_count",
    "campaign_complaint_count",
    "campaign_phishing_count",
    "campaign_moderation_rejected_count"
]

outlier_report = []

for col in outlier_columns:

    q1 = df[col].quantile(0.25)
    q3 = df[col].quantile(0.75)

    iqr = q3 - q1

    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr

    outlier_count = (
        (df[col] < lower_bound) |
        (df[col] > upper_bound)
    ).sum()

    outlier_report.append({
        "column": col,
        "q1": q1,
        "q3": q3,
        "iqr": iqr,
        "lower_bound": lower_bound,
        "upper_bound": upper_bound,
        "outlier_count": outlier_count,
        "outlier_percentage": round(
            outlier_count / len(df) * 100, 2
        )
    })

outlier_report = pd.DataFrame(outlier_report)

display(
    outlier_report.sort_values(
        "outlier_percentage",
        ascending=False
    )
)

,column,q1,q3,iqr,lower_bound,upper_bound,outlier_count,outlier_percentage
8,campaign_subscribers_count,69.0,3353.0,3284.0,-4857.0,8279.0,140224,15.44
10,campaign_subscribers_blocked_count,0.0,0.0,0.0,0.0,0.0,139134,15.32
16,campaign_hard_bounced_count,0.0,8.0,8.0,-12.0,20.0,136247,15.00
15,campaign_soft_bounced_count,0.0,3.0,3.0,-4.5,7.5,131128,14.44
13,campaign_clicks_count,0.0,59.0,59.0,-88.5,147.5,123167,13.56
14,campaign_unique_clicks_count,0.0,33.0,33.0,-49.5,82.5,120341,13.25
12,campaign_unique_opens_count,17.0,472.0,455.0,-665.5,1154.5,111480,12.27
11,campaign_opens_count,29.0,706.0,677.0,-986.5,1721.5,106556,11.73
2,message_embedded_images_count,0.0,0.0,0.0,0.0,0.0,48637,5.36
3,message_embedded_images_size,0.0,0.0,0.0,0.0,0.0,48636,5.36


In [ ]:
# ============================================
# EXTREME VALUE SUMMARY
# ============================================

extreme_summary = df[outlier_columns].describe().T

extreme_summary["skewness"] = (
    df[outlier_columns]
    .skew()
)

display(extreme_summary[
    [
        "count",
        "mean",
        "50%",
        "75%",
        "max",
        "skewness"
    ]
].sort_values("skewness", ascending=False))

,count,mean,50%,75%,max,skewness
message_links_blocked_count,908226.0,0.000002,0.0,0.0,1.0,673.877956
campaign_phantom_bounce_count,908226.0,0.175141,0.0,0.0,24583.0,491.557768
campaign_soft_bounced_count,908226.0,22.336356,0.0,3.0,303995.0,265.172950
campaign_hard_bounced_count,908226.0,43.175298,1.0,8.0,496041.0,255.294221
campaign_complaint_count,908226.0,0.001854,0.0,0.0,29.0,142.230735
campaign_subscribers_blocked_count,908226.0,19.733508,0.0,0.0,113220.0,140.350737
campaign_opens_count,908226.0,900.979523,186.0,706.0,1362091.0,88.183906
campaign_phishing_count,908226.0,0.009500,0.0,0.0,57.0,78.647447
campaign_unique_opens_count,908226.0,641.175196,118.0,472.0,896517.0,78.449231
campaign_clicks_count,908226.0,103.055121,9.0,59.0,106532.0,48.068122


In [ ]:
# ============================================
# COUNT CONSISTENCY AUDIT
# ============================================

consistency_checks = pd.DataFrame({

    "soft_bounce_gt_subscribers":
        df["campaign_soft_bounced_count"]
        > df["campaign_subscribers_count"],

    "hard_bounce_gt_subscribers":
        df["campaign_hard_bounced_count"]
        > df["campaign_subscribers_count"],

    "blocked_gt_subscribers":
        df["campaign_subscribers_blocked_count"]
        > df["campaign_subscribers_count"],

    "phantom_bounce_gt_subscribers":
        df["campaign_phantom_bounce_count"]
        > df["campaign_subscribers_count"],

    "unique_opens_gt_subscribers":
        df["campaign_unique_opens_count"]
        > df["campaign_subscribers_count"],

    "unique_clicks_gt_subscribers":
        df["campaign_unique_clicks_count"]
        > df["campaign_subscribers_count"],

    "complaints_gt_subscribers":
        df["campaign_complaint_count"]
        > df["campaign_subscribers_count"],

    "phishing_gt_subscribers":
        df["campaign_phishing_count"]
        > df["campaign_subscribers_count"]
})

print(consistency_checks.sum())

soft_bounce_gt_subscribers         2
hard_bounce_gt_subscribers         4
blocked_gt_subscribers           144
phantom_bounce_gt_subscribers      0
unique_opens_gt_subscribers        7
unique_clicks_gt_subscribers       6
complaints_gt_subscribers          0
phishing_gt_subscribers            0
dtype: int64


In [ ]:
# 1. Outlier report
display(
    outlier_report.sort_values(
        "outlier_percentage",
        ascending=False
    )
)

,column,q1,q3,iqr,lower_bound,upper_bound,outlier_count,outlier_percentage
8,campaign_subscribers_count,69.0,3353.0,3284.0,-4857.0,8279.0,140224,15.44
10,campaign_subscribers_blocked_count,0.0,0.0,0.0,0.0,0.0,139134,15.32
16,campaign_hard_bounced_count,0.0,8.0,8.0,-12.0,20.0,136247,15.00
15,campaign_soft_bounced_count,0.0,3.0,3.0,-4.5,7.5,131128,14.44
13,campaign_clicks_count,0.0,59.0,59.0,-88.5,147.5,123167,13.56
14,campaign_unique_clicks_count,0.0,33.0,33.0,-49.5,82.5,120341,13.25
12,campaign_unique_opens_count,17.0,472.0,455.0,-665.5,1154.5,111480,12.27
11,campaign_opens_count,29.0,706.0,677.0,-986.5,1721.5,106556,11.73
2,message_embedded_images_count,0.0,0.0,0.0,0.0,0.0,48637,5.36
3,message_embedded_images_size,0.0,0.0,0.0,0.0,0.0,48636,5.36


In [ ]:
# 2. Extreme-value / skewness summary
display(
    extreme_summary[
        [
            "count",
            "mean",
            "50%",
            "75%",
            "max",
            "skewness"
        ]
    ].sort_values(
        "skewness",
        ascending=False
    )
)

,count,mean,50%,75%,max,skewness
message_links_blocked_count,908226.0,0.000002,0.0,0.0,1.0,673.877956
campaign_phantom_bounce_count,908226.0,0.175141,0.0,0.0,24583.0,491.557768
campaign_soft_bounced_count,908226.0,22.336356,0.0,3.0,303995.0,265.172950
campaign_hard_bounced_count,908226.0,43.175298,1.0,8.0,496041.0,255.294221
campaign_complaint_count,908226.0,0.001854,0.0,0.0,29.0,142.230735
campaign_subscribers_blocked_count,908226.0,19.733508,0.0,0.0,113220.0,140.350737
campaign_opens_count,908226.0,900.979523,186.0,706.0,1362091.0,88.183906
campaign_phishing_count,908226.0,0.009500,0.0,0.0,57.0,78.647447
campaign_unique_opens_count,908226.0,641.175196,118.0,472.0,896517.0,78.449231
campaign_clicks_count,908226.0,103.055121,9.0,59.0,106532.0,48.068122


In [ ]:
# ============================================
# FINAL LOGICAL CONSISTENCY CHECK
# Only campaigns with subscribers > 0
# ============================================

positive_audience = df[
    df["campaign_subscribers_count"] > 0
].copy()

sent = positive_audience["campaign_subscribers_count"]

checks = pd.DataFrame({

    "soft_bounce_gt_sent":
        positive_audience["campaign_soft_bounced_count"] > sent,

    "hard_bounce_gt_sent":
        positive_audience["campaign_hard_bounced_count"] > sent,

    "blocked_gt_sent":
        positive_audience["campaign_subscribers_blocked_count"] > sent,

    "phantom_bounce_gt_sent":
        positive_audience["campaign_phantom_bounce_count"] > sent,

    "unique_opens_gt_sent":
        positive_audience["campaign_unique_opens_count"] > sent,

    "unique_clicks_gt_sent":
        positive_audience["campaign_unique_clicks_count"] > sent,

    "complaints_gt_sent":
        positive_audience["campaign_complaint_count"] > sent,

    "phishing_gt_sent":
        positive_audience["campaign_phishing_count"] > sent
})

print("Positive-audience records:", len(positive_audience))
print()
print(checks.sum())

Positive-audience records: 870013

soft_bounce_gt_sent         0
hard_bounce_gt_sent         0
blocked_gt_sent           144
phantom_bounce_gt_sent      0
unique_opens_gt_sent        0
unique_clicks_gt_sent       0
complaints_gt_sent          0
phishing_gt_sent            0
dtype: int64


In [ ]:
blocked_anomalies = positive_audience[
    positive_audience["campaign_subscribers_blocked_count"]
    > positive_audience["campaign_subscribers_count"]
]

print("Blocked > subscribers:", len(blocked_anomalies))

display(
    blocked_anomalies[
        [
            "campaign_id",
            "client_id",
            "campaign_subscribers_count",
            "campaign_subscribers_blocked_count",
            "campaign_phantom_bounce_count",
            "campaign_soft_bounced_count",
            "campaign_hard_bounced_count",
            "campaign_opens_count",
            "campaign_clicks_count",
            "campaign_complaint_count",
            "campaign_sent_time"
        ]
    ].head(30)
)

Blocked > subscribers: 144


,campaign_id,client_id,campaign_subscribers_count,campaign_subscribers_blocked_count,campaign_phantom_bounce_count,campaign_soft_bounced_count,campaign_hard_bounced_count,campaign_opens_count,campaign_clicks_count,campaign_complaint_count,campaign_sent_time
2657,5999013,108092,9410,90735,0,17,54,1154,80,0,2024-04-03 12:20:33+00:00
10876,6003237,4887,1,2,0,0,0,13,1,0,2024-04-08 10:12:27+00:00
10986,6004085,4887,1,2,0,0,0,2,1,0,2024-04-08 11:22:10+00:00
10996,6004113,4887,1,2,0,0,0,4,0,0,2024-04-08 11:30:26+00:00
11019,6003332,4887,1,2,0,0,0,18,20,0,2024-04-08 11:44:24+00:00
12182,6005541,4887,1,2,0,0,0,9,4,0,2024-04-09 10:11:57+00:00
12195,6005548,4887,1,2,0,0,0,22,22,0,2024-04-09 10:19:28+00:00
19466,6005941,108092,8821,92573,0,32,69,705,44,0,2024-04-10 07:01:08+00:00
19739,6006711,4887,1,2,0,0,0,6,2,0,2024-04-10 09:21:00+00:00
19935,6007172,4887,1,2,0,0,0,16,5,0,2024-04-10 11:03:17+00:00


In [ ]:
blocked_anomalies = blocked_anomalies.copy()

blocked_anomalies["blocked_minus_subscribers"] = (
    blocked_anomalies["campaign_subscribers_blocked_count"]
    - blocked_anomalies["campaign_subscribers_count"]
)

print(
    blocked_anomalies[
        [
            "campaign_subscribers_count",
            "campaign_subscribers_blocked_count",
            "blocked_minus_subscribers"
        ]
    ].describe()
)

       campaign_subscribers_count  campaign_subscribers_blocked_count  \
count                  144.000000                          144.000000   
mean                  1960.409722                        19479.840278   
std                   4130.230046                        38015.206439   
min                      1.000000                            2.000000   
25%                      1.000000                            2.000000   
50%                      1.000000                            2.000000   
75%                   1360.000000                        10549.750000   
max                  32632.000000                       113220.000000   

       blocked_minus_subscribers  
count                 144.000000  
mean                17519.430556  
std                 34774.423888  
min                     1.000000  
25%                     1.000000  
50%                     1.000000  
75%                  9369.500000  
max                105262.000000  


In [ ]:
# ============================================
# BLOCKED SUBSCRIBERS ANOMALY AUDIT
# ============================================

blocked_anomalies = positive_audience[
    positive_audience["campaign_subscribers_blocked_count"]
    > positive_audience["campaign_subscribers_count"]
].copy()

print("Blocked > subscribers:", len(blocked_anomalies))

display(
    blocked_anomalies[
        [
            "campaign_id",
            "client_id",
            "campaign_subscribers_count",
            "campaign_subscribers_blocked_count",
            "campaign_phantom_bounce_count",
            "campaign_soft_bounced_count",
            "campaign_hard_bounced_count",
            "campaign_opens_count",
            "campaign_clicks_count",
            "campaign_complaint_count",
            "campaign_sent_time"
        ]
    ].head(30)
)

Blocked > subscribers: 144


,campaign_id,client_id,campaign_subscribers_count,campaign_subscribers_blocked_count,campaign_phantom_bounce_count,campaign_soft_bounced_count,campaign_hard_bounced_count,campaign_opens_count,campaign_clicks_count,campaign_complaint_count,campaign_sent_time
2657,5999013,108092,9410,90735,0,17,54,1154,80,0,2024-04-03 12:20:33+00:00
10876,6003237,4887,1,2,0,0,0,13,1,0,2024-04-08 10:12:27+00:00
10986,6004085,4887,1,2,0,0,0,2,1,0,2024-04-08 11:22:10+00:00
10996,6004113,4887,1,2,0,0,0,4,0,0,2024-04-08 11:30:26+00:00
11019,6003332,4887,1,2,0,0,0,18,20,0,2024-04-08 11:44:24+00:00
12182,6005541,4887,1,2,0,0,0,9,4,0,2024-04-09 10:11:57+00:00
12195,6005548,4887,1,2,0,0,0,22,22,0,2024-04-09 10:19:28+00:00
19466,6005941,108092,8821,92573,0,32,69,705,44,0,2024-04-10 07:01:08+00:00
19739,6006711,4887,1,2,0,0,0,6,2,0,2024-04-10 09:21:00+00:00
19935,6007172,4887,1,2,0,0,0,16,5,0,2024-04-10 11:03:17+00:00


In [ ]:
blocked_anomalies["blocked_minus_subscribers"] = (
    blocked_anomalies["campaign_subscribers_blocked_count"]
    - blocked_anomalies["campaign_subscribers_count"]
)

print(
    blocked_anomalies[
        [
            "campaign_subscribers_count",
            "campaign_subscribers_blocked_count",
            "blocked_minus_subscribers"
        ]
    ].describe()
)

       campaign_subscribers_count  campaign_subscribers_blocked_count  \
count                  144.000000                          144.000000   
mean                  1960.409722                        19479.840278   
std                   4130.230046                        38015.206439   
min                      1.000000                            2.000000   
25%                      1.000000                            2.000000   
50%                      1.000000                            2.000000   
75%                   1360.000000                        10549.750000   
max                  32632.000000                       113220.000000   

       blocked_minus_subscribers  
count                 144.000000  
mean                17519.430556  
std                 34774.423888  
min                     1.000000  
25%                     1.000000  
50%                     1.000000  
75%                  9369.500000  
max                105262.000000  


In [ ]:
print(
    blocked_anomalies[
        [
            "campaign_subscribers_count",
            "campaign_subscribers_blocked_count"
        ]
    ]
    .sort_values(
        "campaign_subscribers_count"
    )
    .head(30)
)

        campaign_subscribers_count  campaign_subscribers_blocked_count
10876                            1                                   2
10986                            1                                   2
10996                            1                                   2
11019                            1                                   2
12195                            1                                   2
12182                            1                                   2
19935                            1                                   2
19739                            1                                   2
86003                            1                                   2
70450                            1                                   2
70470                            1                                   2
101154                           1                                   2
101302                           1                                   2
93136 

In [ ]:
# ============================================
# FINAL DATA INTEGRITY AUDIT
# ============================================

print("Rows:", len(df))
print("Columns:", len(df.columns))

print("\nExact duplicate rows:")
print(df.duplicated().sum())

print("\nMissing campaign timestamps:")
print(df["campaign_sent_time"].isna().sum())

print("\nNegative numeric values:")

numeric_cols = df.select_dtypes(include="number").columns

negative_summary_final = (
    (df[numeric_cols] < 0)
    .sum()
)

print(
    negative_summary_final[
        negative_summary_final > 0
    ]
)

print("\nZero-subscriber campaigns:")
print(
    (df["campaign_subscribers_count"] == 0).sum()
)

print("\nZero-subscriber campaigns with activity:")
print(
    df["zero_audience_with_activity"].sum()
)

print("\nDomain negative sentinel values:")

for col in negative_domain_cols:
    print(
        col,
        "negative:",
        (df[col] < 0).sum()
    )

print("\nBlocked > subscribers:")
print(
    (
        df["campaign_subscribers_blocked_count"]
        > df["campaign_subscribers_count"]
    ).sum()
)

Rows: 908226
Columns: 136

Exact duplicate rows:
0

Missing campaign timestamps:
0

Negative numeric values:
Series([], dtype: int64)

Zero-subscriber campaigns:
38213

Zero-subscriber campaigns with activity:
7

Domain negative sentinel values:
domain_in_links_min_creation_time_diff negative: 0
domain_in_links_max_creation_time_diff negative: 0
domain_in_links_avg_creation_time_diff negative: 0
domain_in_links_most_used_creation_time_diff negative: 0

Blocked > subscribers:
144


In [ ]:
# ============================================
# CLEANING SUMMARY
# ============================================

cleaning_summary = pd.DataFrame({
    "check": [
        "Raw rows",
        "Exact duplicate rows removed",
        "Final rows",
        "Invalid timestamps",
        "Domain -1 sentinel values",
        "Zero-subscriber records",
        "Zero-subscriber records with activity",
        "Blocked > subscribers records"
    ],
    "result": [
        908229,
        3,
        len(df),
        df["campaign_sent_time"].isna().sum(),
        0,
        (df["campaign_subscribers_count"] == 0).sum(),
        df["zero_audience_with_activity"].sum(),
        (
            df["campaign_subscribers_blocked_count"]
            > df["campaign_subscribers_count"]
        ).sum()
    ],
    "action": [
        "Source dataset",
        "Removed exact duplicates",
        "Final processed dataset",
        "All timestamps successfully parsed",
        "Converted -1 sentinel to NaN",
        "Retained and flagged",
        "Retained and flagged",
        "Retained; valid because blocked and sent are different populations"
    ]
})

display(cleaning_summary)

,check,result,action
0,Raw rows,908229,Source dataset
1,Exact duplicate rows removed,3,Removed exact duplicates
2,Final rows,908226,Final processed dataset
3,Invalid timestamps,0,All timestamps successfully parsed
4,Domain -1 sentinel values,0,Converted -1 sentinel to NaN
5,Zero-subscriber records,38213,Retained and flagged
6,Zero-subscriber records with activity,7,Retained and flagged
7,Blocked > subscribers records,144,Retained; valid because blocked and sent are d...


In [ ]:
cleaning_summary_path = (
    "/content/drive/MyDrive/"
    "AI_Email_Deliverability_Intelligence/"
    "reports/base_data_cleaning_summary.csv"
)

cleaning_summary.to_csv(
    cleaning_summary_path,
    index=False
)

print("Saved:", cleaning_summary_path)

Saved: /content/drive/MyDrive/AI_Email_Deliverability_Intelligence/reports/base_data_cleaning_summary.csv


In [ ]:
# ============================================
# FINAL CLEAN BASE DATASET
# ============================================

clean_base_path = (
    "/content/drive/MyDrive/"
    "AI_Email_Deliverability_Intelligence/"
    "data/processed/"
    "sendguard_base_clean.parquet"
)

df.to_parquet(
    clean_base_path,
    index=False
)

print("Final clean base dataset saved:")
print(clean_base_path)

Final clean base dataset saved:
/content/drive/MyDrive/AI_Email_Deliverability_Intelligence/data/processed/sendguard_base_clean.parquet


In [ ]:
check_clean = pd.read_parquet(clean_base_path)

print("Final shape:", check_clean.shape)
print("Exact duplicates:", check_clean.duplicated().sum())
print(
    "Missing timestamps:",
    check_clean["campaign_sent_time"].isna().sum()
)

Final shape: (908226, 136)
Exact duplicates: 0
Missing timestamps: 0
